In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Data Cleaning environment ready!")

## 1. Load Raw Match Records
We load `raw_data.csv`, which aggregates individual season CSVs downloaded from Football-Data.co.uk.

In [ ]:
# Resilient path resolution (works from root or docs/ directory)
candidate_paths = [
    Path("raw_data.csv"),
    Path("data/raw_data.csv"),
    Path("../raw_data.csv"),
    Path("../data/raw_data.csv")
]
raw_path = next((p for p in candidate_paths if p.exists()), Path("raw_data.csv"))
print(f"Loading raw data from: {raw_path.resolve()}")

df_raw = pd.read_csv(raw_path, low_memory=False, encoding="latin1")
print(f"Raw Dataset Shape: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns")
df_raw.head()

## 2. Inspecting Raw Data Quality & Missingness
In sports analytics, historical datasets often have missing entries because stats like match shots, fouls, or certain bookmaker odds were only recorded in later years.

In [ ]:
missing_counts = df_raw.isnull().sum()
missing_pct = (missing_counts / len(df_raw)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_counts, 'Missing_Percent': missing_pct})
missing_df = missing_df.sort_values(by='Missing_Percent', ascending=False)

print("Top 15 Columns with Highest Missing Percentages:")
missing_df.head(15)

In [ ]:
core_cols = ['Date', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR', 'HS', 'AS', 'HST', 'AST', 'B365H', 'B365D', 'B365A']
core_missing = missing_df.loc[missing_df.index.isin(core_cols)]

plt.figure(figsize=(10, 4))
sns.barplot(x=core_missing.index, y=core_missing['Missing_Percent'], palette='Blues_r')
plt.title("Missing Data Percentage in Core Match Attributes (%)", fontsize=13, fontweight='bold')
plt.ylabel("Missing %")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Schema Standardization & Renaming
Column names in raw football data frequently change across decades (`FTHG` vs `HomeGoals`, `FTR` vs `Result`). We map them to canonical names:
- `Date` $\rightarrow$ `MatchDate`
- `FTHG` $\rightarrow$ `FTHome` (Full-Time Home Goals)
- `FTAG` $\rightarrow$ `FTAway` (Full-Time Away Goals)
- `FTR` $\rightarrow$ `FTResult` (Match Outcome: H, D, A)
- `HS` / `AS` $\rightarrow$ `HomeShots` / `AwayShots`
- `HST` / `AST` $\rightarrow$ `HomeShotsTarget` / `AwayShotsTarget`

In [ ]:
df_clean = df_raw.copy()

col_map = {
    "Date": "MatchDate",
    "FTHG": "FTHome",
    "FTAG": "FTAway",
    "FTR": "FTResult",
    "HS": "HomeShots",
    "AS": "AwayShots",
    "HST": "HomeShotsTarget",
    "AST": "AwayShotsTarget",
}

for old, new in col_map.items():
    if old in df_clean.columns and new not in df_clean.columns:
        df_clean[new] = df_clean[old]

print("Schema standardized. Core columns confirmed:")
print([col for col in col_map.values() if col in df_clean.columns])

## 4. Multi-Bookmaker Odds Consolidation & Fallback Logic
Over 20 seasons, individual sportsbooks entered or exited the market. If we rely solely on Bet365 (`B365`), matches from older seasons or specific leagues will have missing odds.
We build an automated fallback chain:
$$\text{Odd} = \text{FirstValid}(\text{Avg}, \text{B365}, \text{WH}, \text{BW}, \text{IW}, \text{PS})$$

In [ ]:
for side, code in [("Home", "H"), ("Draw", "D"), ("Away", "A")]:
    out_col = f"Odd{side}"
    candidates = [f"Avg{code}", f"B365{code}", f"WH{code}", f"BW{code}", f"IW{code}", f"PS{code}"]
    extracted = None
    for cand in candidates:
        if cand in df_clean.columns:
            val = pd.to_numeric(df_clean[cand], errors="coerce")
            extracted = val if extracted is None else extracted.fillna(val)
    df_clean[out_col] = extracted

print("Consolidated Odds Null Counts:")
print(df_clean[['OddHome', 'OddDraw', 'OddAway']].isnull().sum())

## 5. Flexible Datetime Parsing & Chronological Ordering
Dates in CSV files often mix 2-digit years (`15/08/05`) with 4-digit years (`15/08/2005`).
We parse them with `dayfirst=True` and sort strictly chronologically. This is vital to prevent future data leakage during feature generation.

In [ ]:
df_clean["MatchDate"] = pd.to_datetime(df_clean["MatchDate"], dayfirst=True, errors="coerce")
unparsed_dates = df_clean["MatchDate"].isnull().sum()
print(f"Unparsed date records: {unparsed_dates}")

# Sort chronologically
df_clean = df_clean.sort_values("MatchDate").reset_index(drop=True)
print(f"Date Range: from {df_clean['MatchDate'].min().strftime('%Y-%m-%d')} to {df_clean['MatchDate'].max().strftime('%Y-%m-%d')}")

## 6. Data Integrity & Anomaly Filtering
We drop unplayed fixtures, abandoned matches, and sanitize target values:
- Ensure `FTHome` and `FTAway` are non-negative integers.
- Ensure `FTResult` is strictly one of `'H'` (Home Win), `'D'` (Draw), or `'A'` (Away Win).

In [ ]:
initial_len = len(df_clean)
df_clean = df_clean.dropna(subset=["MatchDate", "HomeTeam", "AwayTeam", "FTHome", "FTAway", "FTResult"])

# Filter valid result tokens
df_clean["FTResult"] = df_clean["FTResult"].astype(str).str.strip().str.upper()
df_clean = df_clean[df_clean["FTResult"].isin(["H", "D", "A"])]

df_clean["FTHome"] = df_clean["FTHome"].astype(int)
df_clean["FTAway"] = df_clean["FTAway"].astype(int)

# Sanity check: no negative goals
df_clean = df_clean[(df_clean["FTHome"] >= 0) & (df_clean["FTAway"] >= 0)]

print(f"Filtered {initial_len - len(df_clean)} incomplete/invalid match rows.")
print(f"Cleaned Valid Matches: {len(df_clean):,}")

## 7. Data Cleaning Verification & Export
The clean base dataset is ready to serve as the single source of truth for EDA and Feature Engineering.

In [ ]:
export_candidates = [Path("data/cleaned_base_matches.csv"), Path("../data/cleaned_base_matches.csv")]
export_path = export_candidates[0] if Path("data").exists() else export_candidates[1]
export_path.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(export_path, index=False)
print(f"Cleaned base dataset successfully saved to: {export_path.resolve()}")
df_clean[['MatchDate', 'LeagueCode', 'HomeTeam', 'AwayTeam', 'FTHome', 'FTAway', 'FTResult', 'OddHome', 'OddDraw', 'OddAway']].head(10)